# Step 1: Find and Catalogue Multi-Sector Targets

**Why this step:** The entire cross-sector consistency idea only works on stars observed more than once. Choosing the target list well up front avoids weeks of wasted effort chasing stars with only single-sector coverage.

This notebook queries NASA's MAST archive for TESS targets with 3+ sectors of observations, prioritizing the Continuous Viewing Zone (CVZ) near the ecliptic poles where stars can accumulate 20+ sectors of repeat coverage.

In [ ]:
from astroquery.mast import Observations
import pandas as pd
import numpy as np

## Config

In [ ]:
MIN_SECTORS = 3
# Minimum number of independent sectors required for a star to be useful
# for cross-sector consistency checking. Below this, "consistency" is
# statistically too weak to claim much.

OUTPUT_CSV = "multisector_tic_targets.csv"
# Where the final target list gets saved -- this becomes the backbone
# that every later pipeline step reads from.

# Approximate CVZ regions, as (RA, Dec) box centers + a search radius.
# The ecliptic poles sit at roughly RA=90, Dec=+/-66.56 -- the 66.56
# comes from 90 - 23.44 (Earth's axial tilt), since the ecliptic pole is
# offset from the celestial pole by exactly the tilt angle.
#
# WHY A BOX AND NOT THE WHOLE SKY: pushing this coordinate filter into
# the query itself keeps the MAST query fast and avoids downloading
# metadata for thousands of irrelevant single-sector stars we'd just
# throw away anyway.
CVZ_REGIONS = [
    {"name": "north_cvz", "ra": 90.0, "dec": 66.56, "radius_deg": 12},
    {"name": "south_cvz", "ra": 90.0, "dec": -66.56, "radius_deg": 12},
]

## Step functions

In [ ]:
def query_cvz_region(ra, dec, radius_deg):
    """
    Query MAST for TESS timeseries observations near a given (ra, dec).
    """
    obs = Observations.query_criteria(
        obs_collection="TESS",
        # Only TESS data -- MAST hosts many missions (Hubble, JWST,
        # Kepler, etc.) and we don't want those mixed in.

        dataproduct_type="timeseries",
        # Light curve products only, not raw images -- keeps the query
        # fast and avoids pulling irrelevant image-product rows.

        s_ra=[ra - radius_deg, ra + radius_deg],
        s_dec=[dec - radius_deg, dec + radius_deg],
        # Simple bounding-box search. Not a perfectly accurate cone
        # search (RA lines converge near the poles, so a naive box gets
        # slightly distorted there), but good enough for building an
        # initial candidate list.
    )
    # Convert from astropy Table to pandas DataFrame -- groupby/
    # filtering/sorting are all much easier in pandas.
    return obs.to_pandas()


def extract_tic_id(target_name):
    """
    Pull a clean, digits-only TIC ID out of MAST's target_name field.

    WHY THIS IS NECESSARY: MAST's target_name column is inconsistently
    formatted -- sometimes "TIC 123456789", sometimes just "123456789",
    sometimes a non-TIC proposal-specific name. Grouping directly on the
    raw string would treat "TIC 123456789" and "123456789" as two
    DIFFERENT stars, silently under-counting that star's real sector
    count.
    """
    if pd.isna(target_name):
        return None
    s = str(target_name).replace("TIC", "").strip()
    return s if s.isdigit() else None

## Step A: Query both CVZ regions and combine into one observation-level table

In [ ]:
# At this point, obs_df will have ONE ROW PER SECTOR-OBSERVATION, not
# per star. If a star was observed in 5 sectors, it appears as 5
# separate rows here.

all_obs = []
for region in CVZ_REGIONS:
    print(f"Querying {region['name']}...")
    df = query_cvz_region(region["ra"], region["dec"], region["radius_deg"])

    # Tag which CVZ region each row came from -- useful for later
    # debugging or analysis.
    df["cvz_region"] = region["name"]

    all_obs.append(df)
    print(f"  -> {len(df)} observation rows")

# ignore_index=True re-numbers rows 0,1,2,... after stacking -- without
# it, the two DataFrames' original indices could overlap/clash.
obs_df = pd.concat(all_obs, ignore_index=True)

## Step B: Extract clean TIC IDs, then collapse to one row per star with a sector count

In [ ]:
obs_df["tic_id"] = obs_df["target_name"].apply(extract_tic_id)

# Drop rows where we couldn't extract a valid TIC ID.
obs_df = obs_df.dropna(subset=["tic_id"])

# THE CORE TRANSFORMATION:
# groupby("tic_id") clusters all rows belonging to the same star together.
# .nunique() counts DISTINCT sector numbers per star.
#
# WHY nunique() AND NOT count(): MAST sometimes has duplicate entries
# for the same sector (different pipeline versions/reprocessings). A
# plain count() would count those duplicates as extra sectors,
# artificially inflating a star's sector count.
sector_counts = (
    obs_df.groupby("tic_id")["sequence_number"]
    .nunique()
    .reset_index()                                    # turn tic_id back into a normal column
    .rename(columns={"sequence_number": "n_sectors"})  # more readable name
)

## Step C: Filter to stars meeting the minimum sector requirement, then sort

In [ ]:
multisector = sector_counts[sector_counts["n_sectors"] >= MIN_SECTORS]

# Sort descending so the highest-coverage stars (true CVZ stars,
# potentially 15-20+ sectors) appear first -- also acts as a quick
# sanity check.
multisector = multisector.sort_values("n_sectors", ascending=False)

print(f"\nFound {len(multisector)} TIC IDs with >= {MIN_SECTORS} sectors")
print(multisector.head(10))

## Step D: Save to disk

In [ ]:
# WHY SAVE NOW: checkpointing (MAST queries can be slow/flaky),
# reproducibility, and decoupling this step from Step 2 (so Step 2 can
# be re-run without re-querying MAST every time).
multisector.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved target list -> {OUTPUT_CSV}")

## Sanity checks

In [ ]:
print(multisector["n_sectors"].describe())
print(multisector["n_sectors"].value_counts().sort_index(ascending=False).head(15))

# Expect to see some stars in the 15-20+ sector range near the top -- if
# the max sector count anywhere is only 3-4, the CVZ coordinate box is
# probably off-target and should be re-checked.